# COMPAS Analysis by Sex

Data and `truth_tables.py` from ProPublica's [compas-analysis](https://github.com/propublica/compas-analysis) repository; code adapted from their analysis.

## Setup

In [ ]:
import os, urllib.request
base = "https://raw.githubusercontent.com/propublica/compas-analysis/master/"
for f in ["compas-scores-two-years.csv", "compas-scores-two-years-violent.csv",
          "cox-parsed.csv", "cox-violent-parsed.csv", "truth_tables.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(base + f, f)
print(sorted(os.listdir(".")))

In [ ]:
# filter dplyr warnings
%load_ext rpy2.ipython
import warnings
warnings.filterwarnings('ignore')

In [ ]:
%%R
suppressWarnings(for (p in c("dplyr", "ggplot2", "gridExtra", "survival", "ggfortify", "car", "pROC")) {
  if (!requireNamespace(p, quietly = TRUE)) install.packages(p)
})

## 1. Data

In [ ]:
%%R
suppressPackageStartupMessages({
  library(dplyr)
  library(ggplot2)
})
raw_data <- read.csv("./compas-scores-two-years.csv")
nrow(raw_data)

In [ ]:
%%R
df <- dplyr::select(raw_data, age, c_charge_degree, race, age_cat, score_text, sex, priors_count,
                    days_b_screening_arrest, decile_score, is_recid, two_year_recid, c_jail_in, c_jail_out) %>%
        filter(days_b_screening_arrest <= 30) %>%
        filter(days_b_screening_arrest >= -30) %>%
        filter(is_recid != -1) %>%
        filter(c_charge_degree != "O") %>%
        filter(score_text != 'N/A')
nrow(df)

In [ ]:
%%R
df$length_of_stay <- as.numeric(as.Date(df$c_jail_out) - as.Date(df$c_jail_in))
cor(df$length_of_stay, df$decile_score)

In [ ]:
%%R
summary(as.factor(df$age_cat))

In [ ]:
%%R
summary(as.factor(df$race))

In [ ]:
%%R
summary(as.factor(df$score_text))

In [ ]:
%%R
xtabs(~ sex + race, data=df)

In [ ]:
%%R
summary(as.factor(df$sex))

In [ ]:
%%R
# share of the filtered sample by sex (computed, not typed in)
round(prop.table(table(df$sex)) * 100, 2)

In [ ]:
%%R
nrow(filter(df, two_year_recid == 1))

In [ ]:
%%R
nrow(filter(df, two_year_recid == 1)) / nrow(df) * 100

In [ ]:
%%R
nrow(filter(df, two_year_recid == 1, sex == "Female")) / nrow(filter(df, sex == "Female")) * 100

In [ ]:
%%R
nrow(filter(df, two_year_recid == 1, sex == "Male")) / nrow(filter(df, sex == "Male")) * 100

## 2. Score distributions

In [ ]:
%%R -w 900 -h 363 -u px
suppressPackageStartupMessages({
  library(grid)
  library(gridExtra)
})
pfemale <- ggplot(data=filter(df, sex == "Female"), aes(ordered(decile_score))) +
          geom_bar() + xlab("Decile Score") +
          ylim(0, 1100) + ggtitle("Female defendants' decile scores")
pmale   <- ggplot(data=filter(df, sex == "Male"), aes(ordered(decile_score))) +
          geom_bar() + xlab("Decile Score") +
          ylim(0, 1100) + ggtitle("Male defendants' decile scores")
grid.arrange(pfemale, pmale,  ncol = 2)

In [ ]:
%%R
xtabs(~ decile_score + sex, data=df)

## 3. Logistic regression: general score

In [ ]:
%%R
df <- mutate(df, crime_factor = factor(c_charge_degree)) %>%
      mutate(age_factor = as.factor(age_cat)) %>%
      within(age_factor <- relevel(age_factor, ref = 1)) %>%
      mutate(race_factor = factor(race)) %>%
      within(race_factor <- relevel(race_factor, ref = 3)) %>%
      mutate(gender_factor = factor(sex, labels= c("Female","Male"))) %>%
      within(gender_factor <- relevel(gender_factor, ref = 2)) %>%
      mutate(score_factor = factor(score_text != "Low", labels = c("LowScore","HighScore")))
model <- glm(score_factor ~ gender_factor + age_factor + race_factor +
                            priors_count + crime_factor + two_year_recid, family="binomial", data=df)
summary(model)

In [ ]:
%%R
# Convert the log-odds coefficient into "X times as likely" using ProPublica's formula.
# Coefficients come straight from the fitted model, so nothing is retyped by hand.
b <- coef(model)
control <- plogis(b[["(Intercept)"]])   # probability of a higher score for the reference group
rel <- function(term) exp(b[[term]]) / (1 - control + control * exp(b[[term]]))
c(female_vs_male = rel("gender_factorFemale"),
  under25_vs_25to45 = rel("age_factorLess than 25"))

## 4. Logistic regression: violent score

In [ ]:
%%R
raw_data <- read.csv("./compas-scores-two-years-violent.csv")
nrow(raw_data)

In [ ]:
%%R
df <- dplyr::select(raw_data, age, c_charge_degree, race, age_cat, v_score_text, sex, priors_count,
                    days_b_screening_arrest, v_decile_score, is_recid, two_year_recid) %>%
        filter(days_b_screening_arrest <= 30) %>%
        filter(days_b_screening_arrest >= -30) %>%
        filter(is_recid != -1) %>%
        filter(c_charge_degree != "O") %>%
        filter(v_score_text != 'N/A')
nrow(df)

In [ ]:
%%R
summary(as.factor(df$sex))

In [ ]:
%%R
summary(as.factor(df$v_score_text))

In [ ]:
%%R
nrow(filter(df, two_year_recid == 1)) / nrow(df) * 100

In [ ]:
%%R -w 900 -h 363 -u px
pfemale <- ggplot(data=filter(df, sex == "Female"), aes(ordered(v_decile_score))) +
          geom_bar() + xlab("Violent Decile Score") +
          ylim(0, 1100) + ggtitle("Female defendants' violent decile scores")
pmale   <- ggplot(data=filter(df, sex == "Male"), aes(ordered(v_decile_score))) +
          geom_bar() + xlab("Violent Decile Score") +
          ylim(0, 1100) + ggtitle("Male defendants' violent decile scores")
grid.arrange(pfemale, pmale,  ncol = 2)

In [ ]:
%%R
df <- mutate(df, crime_factor = factor(c_charge_degree)) %>%
      mutate(age_factor = as.factor(age_cat)) %>%
      within(age_factor <- relevel(age_factor, ref = 1)) %>%
      mutate(race_factor = factor(race,
                                  labels = c("African-American",
                                             "Asian",
                                             "Caucasian",
                                             "Hispanic",
                                             "Native American",
                                             "Other"))) %>%
      within(race_factor <- relevel(race_factor, ref = 3)) %>%
      mutate(gender_factor = factor(sex, labels= c("Female","Male"))) %>%
      within(gender_factor <- relevel(gender_factor, ref = 2)) %>%
      mutate(score_factor = factor(v_score_text != "Low", labels = c("LowScore","HighScore")))
model <- glm(score_factor ~ gender_factor + age_factor + race_factor +
                            priors_count + crime_factor + two_year_recid, family="binomial", data=df)
summary(model)

In [ ]:
%%R
# Convert the log-odds coefficient into "X times as likely" using ProPublica's formula.
# Coefficients come straight from the fitted model, so nothing is retyped by hand.
b <- coef(model)
control <- plogis(b[["(Intercept)"]])   # probability of a higher score for the reference group
rel <- function(term) exp(b[[term]]) / (1 - control + control * exp(b[[term]]))
rel("gender_factorFemale")

## 5. Cox models and survival curves: general score

In [ ]:
%%R
suppressPackageStartupMessages({
  library(survival)
  library(ggfortify)
})

data <- filter(filter(read.csv("./cox-parsed.csv"), score_text != "N/A"), end > start) %>%
        mutate(race_factor = factor(race,
                                  labels = c("African-American",
                                             "Asian",
                                             "Caucasian",
                                             "Hispanic",
                                             "Native American",
                                             "Other"))) %>%
        within(race_factor <- relevel(race_factor, ref = 3)) %>%
        mutate(score_factor = factor(score_text)) %>%
        within(score_factor <- relevel(score_factor, ref=2)) %>%
        mutate(gender_factor = factor(sex, labels= c("Female","Male"))) %>%
        within(gender_factor <- relevel(gender_factor, ref = 2))

grp <- data[!duplicated(data$id),]
nrow(grp)

In [ ]:
%%R
summary(grp$score_factor)

In [ ]:
%%R
summary(grp$gender_factor)

In [ ]:
%%R
f <- Surv(start, end, event, type="counting") ~ score_factor
model <- coxph(f, data=data)
summary(model)

In [ ]:
%%R
decile_f <- Surv(start, end, event, type="counting") ~ decile_score
dmodel <- coxph(decile_f, data=data)
summary(dmodel)

In [ ]:
%%R
f2 <- Surv(start, end, event, type="counting") ~ gender_factor + score_factor + gender_factor * score_factor
model <- coxph(f2, data=data)
print(summary(model))

**Proportional hazards check.** A Cox model assumes each hazard ratio stays the same over the whole follow-up period. `cox.zph` tests that for each term: a small p-value means the ratio changes over time. With samples this large, even small drifts can come out significant, so look at the plot too: a roughly flat line means the assumption is reasonable.

In [ ]:
%%R -w 900 -h 500 -u px
# Test the proportional-hazards assumption for the general-score interaction model
zph <- cox.zph(model)
print(zph)
par(mfrow = c(1, 2))
plot(zph[1], main = "Female (vs. male)")
plot(zph[2], main = "Score label")

In [ ]:
%%R
# Hazard ratio vs. a low-risk defendant of the same sex.
# For women, add the interaction term to the main effect.
b <- coef(model)
round(exp(c(
  female_high   = b[["score_factorHigh"]]   + b[["gender_factorFemale:score_factorHigh"]],
  male_high     = b[["score_factorHigh"]],
  female_medium = b[["score_factorMedium"]] + b[["gender_factorFemale:score_factorMedium"]],
  male_medium   = b[["score_factorMedium"]]
)), 2)

In [ ]:
%%R -w 900 -h 563 -u px

fit <- survfit(f, data=data)

plotty <- function(fit, title) {
  return(autoplot(fit, conf.int=T, censor=F) + ggtitle(title) + ylim(0,1))
}
suppressMessages(print(plotty(fit, "Overall")))

In [ ]:
%%R -w 900 -h 363 -u px
female <- filter(data, sex == "Female")
female_fit <- survfit(f, data=female)

male <- filter(data, sex == "Male")
male_fit <- survfit(f, data=male)

suppressMessages(grid.arrange(plotty(female_fit, "Female defendants"),
             plotty(male_fit, "Male defendants"), ncol=2))

In [ ]:
%%R
summary(fit, times=c(730))

In [ ]:
%%R
summary(female_fit, times=c(730))

In [ ]:
%%R
summary(male_fit, times=c(730))

In [ ]:
%%R
summary(coxph(f, data=female))

In [ ]:
%%R
summary(coxph(f, data=male))

## 6. Cox models and survival curves: violent score

In [ ]:
%%R
violent_data <- filter(filter(read.csv("./cox-violent-parsed.csv"), score_text != "N/A"), end > start) %>%
        mutate(race_factor = factor(race,
                                  labels = c("African-American",
                                             "Asian",
                                             "Caucasian",
                                             "Hispanic",
                                             "Native American",
                                             "Other"))) %>%
        within(race_factor <- relevel(race_factor, ref = 3)) %>%
        mutate(score_factor = factor(score_text)) %>%
        within(score_factor <- relevel(score_factor, ref=2)) %>%
        mutate(gender_factor = factor(sex, labels= c("Female","Male"))) %>%
        within(gender_factor <- relevel(gender_factor, ref = 2))


vf <- Surv(start, end, event, type="counting") ~ score_factor
vmodel <- coxph(vf, data=violent_data)
vgrp <- violent_data[!duplicated(violent_data$id),]
print(nrow(vgrp))
summary(vmodel)

In [ ]:
%%R
vf2 <- Surv(start, end, event, type="counting") ~ gender_factor + gender_factor * score_factor
vmodel <- coxph(vf2, data=violent_data)
summary(vmodel)

**Proportional hazards check (violent score).**

In [ ]:
%%R
cox.zph(vmodel)

**Women vs. men with the same violent label.** The main effect gives the low-risk comparison; for medium and high, the interaction term is added. The confidence interval for a sum of coefficients uses their variances *and* covariance, which is what `vcov` supplies.

In [ ]:
%%R
vb <- coef(vmodel); vV <- vcov(vmodel)
hr_ci <- function(terms) {
  est <- sum(vb[terms]); se <- sqrt(sum(vV[terms, terms]))
  round(c(hr = exp(est), lo = exp(est - 1.96 * se), hi = exp(est + 1.96 * se)), 3)
}
fem <- "gender_factorFemale"
rbind(Low    = hr_ci(fem),
      Medium = hr_ci(c(fem, "gender_factorFemale:score_factorMedium")),
      High   = hr_ci(c(fem, "gender_factorFemale:score_factorHigh")))

In [ ]:
%%R
summary(coxph(vf, data=filter(violent_data, sex == "Female")))

In [ ]:
%%R
summary(coxph(vf, data=filter(violent_data, sex == "Male")))

In [ ]:
%%R -w 900 -h 363 -u px
vfemale <- filter(violent_data, sex == "Female")
vfemale_fit <- survfit(vf, data=vfemale)

vmale <- filter(violent_data, sex == "Male")
vmale_fit <- survfit(vf, data=vmale)

suppressMessages(grid.arrange(plotty(vfemale_fit, "Female defendants"),
             plotty(vmale_fit, "Male defendants"), ncol=2))

## 7. False positive and false negative rates: general score

In [ ]:
from truth_tables import PeekyReader, Person, table, is_race, count, vtable, hightable, vhightable
from csv import DictReader

people = []
with open("./cox-parsed.csv") as f:
    reader = PeekyReader(DictReader(f))
    try:
        while True:
            p = Person(reader)
            if p.valid:
                people.append(p)
    except StopIteration:
        pass

pop = list(filter(lambda i: ((i.recidivist == True and i.lifetime <= 730) or
                              i.lifetime > 730), list(filter(lambda x: x.score_valid, people))))
recid = list(filter(lambda i: i.recidivist == True and i.lifetime <= 730, pop))
rset = set(recid)
surv = [i for i in pop if i not in rset]

In [ ]:
print("All defendants")
table(list(recid), list(surv))

In [ ]:
import statistics
print("Average followup time %.2f (sd %.2f)" % (statistics.mean(map(lambda i: i.lifetime, pop)),
                                                statistics.stdev(map(lambda i: i.lifetime, pop))))
print("Median followup time %i" % (statistics.median(map(lambda i: i.lifetime, pop))))

In [ ]:
def is_sex(sex):
    return lambda x: x.rows[0]['sex'] == sex

In [ ]:
print("Female defendants")
is_female = is_sex("Female")
table(list(filter(is_female, recid)), list(filter(is_female, surv)))

In [ ]:
print("Male defendants")
is_male = is_sex("Male")
table(list(filter(is_male, recid)), list(filter(is_male, surv)))

In [ ]:
# Female-to-male ratios of the error rates, computed from the same groups as the tables above
def fpr(group):
    s = [i for i in surv if group(i)]
    return count(lambda i: i.high, s) / len(s) * 100

def fnr(group):
    r = [i for i in recid if group(i)]
    return count(lambda i: i.low, r) / len(r) * 100

print("FPR ratio (female / male): %.2f" % (fpr(is_female) / fpr(is_male)))
print("FNR ratio (female / male): %.2f" % (fnr(is_female) / fnr(is_male)))

In [ ]:
print("Female")
hightable(list(filter(is_female, recid)), list(filter(is_female, surv)))
print("Male")
hightable(list(filter(is_male, recid)), list(filter(is_male, surv)))

## 8. False positive and false negative rates: violent score

In [ ]:
vpeople = []
with open("./cox-violent-parsed.csv") as f:
    reader = PeekyReader(DictReader(f))
    try:
        while True:
            p = Person(reader)
            if p.valid:
                vpeople.append(p)
    except StopIteration:
        pass

vpop = list(filter(lambda i: ((i.violent_recidivist == True and i.lifetime <= 730) or
                              i.lifetime > 730), list(filter(lambda x: x.vscore_valid, vpeople))))
vrecid = list(filter(lambda i: i.violent_recidivist == True and i.lifetime <= 730, vpeople))
vrset = set(vrecid)
vsurv = [i for i in vpop if i not in vrset]

In [ ]:
print("All defendants")
vtable(list(vrecid), list(vsurv))

In [ ]:
print("Female defendants")
vtable(list(filter(is_female, vrecid)), list(filter(is_female, vsurv)))

In [ ]:
print("Male defendants")
vtable(list(filter(is_male, vrecid)), list(filter(is_male, vsurv)))

## 9. Cox model with decile score interaction and controls

In [ ]:
%%R
suppressPackageStartupMessages({
  library(survival)
  library(dplyr)
})

data <- filter(filter(read.csv("./cox-parsed.csv"), score_text != "N/A"), end > start) %>%
        mutate(race_factor = factor(race,
                                  labels = c("African-American",
                                             "Asian",
                                             "Caucasian",
                                             "Hispanic",
                                             "Native American",
                                             "Other"))) %>%
        within(race_factor <- relevel(race_factor, ref = 3))

grp <- data[!duplicated(data$id),]
nrow(grp)
f <- Surv(start, end, event, type="counting") ~ gender_factor * decile_score + age_factor + race_factor +
                                                 priors_count + crime_factor
felonies <- c('(F1)','(F2)', '(F3)','(F5)','(F6)','(F7)')
data_filtered <-  mutate(data, crime_factor = factor(ifelse(c_charge_degree %in% felonies, 'F', 'M'))) %>%
                  mutate(age_factor = as.factor(age_cat)) %>%
                  within(age_factor <- relevel(age_factor, ref = 1)) %>%
                  mutate(gender_factor = factor(sex, labels= c("Female","Male"))) %>%
                  within(gender_factor <- relevel(gender_factor, ref = 2)) %>%
                  filter(days_b_screening_arrest <= 30) %>%
                  filter(days_b_screening_arrest >= -30) %>%
                  filter(is_recid != -1)
grp <- data_filtered[!duplicated(data_filtered$id),]
print(nrow(grp))
model <- coxph(f, data=data_filtered)
print(summary(model))

**Proportional hazards check (decile model with controls).**

In [ ]:
%%R
cox.zph(model)

In [ ]:
%%R
# Female vs. male hazard ratio at each decile score: main effect + interaction x decile
b <- coef(model)
d <- 1:10
data.frame(decile_score = d,
           female_vs_male_hazard_ratio = round(exp(b[["gender_factorFemale"]] +
                                                   b[["gender_factorFemale:decile_score"]] * d), 3))

## 10. Logistic regression without two-year recidivism

In [ ]:
%%R
raw_data <- read.csv("./compas-scores-two-years.csv")

df <- dplyr::select(raw_data, age, c_charge_degree, race, age_cat, score_text, sex, priors_count,
                    days_b_screening_arrest, decile_score, is_recid, two_year_recid, c_jail_in, c_jail_out) %>%
        filter(days_b_screening_arrest <= 30) %>%
        filter(days_b_screening_arrest >= -30) %>%
        filter(is_recid != -1) %>%
        filter(c_charge_degree != "O") %>%
        filter(score_text != 'N/A')

df <- mutate(df, crime_factor = factor(c_charge_degree)) %>%
      mutate(age_factor = as.factor(age_cat)) %>%
      within(age_factor <- relevel(age_factor, ref = 1)) %>%
      mutate(race_factor = factor(race)) %>%
      within(race_factor <- relevel(race_factor, ref = 3)) %>%
      mutate(gender_factor = factor(sex, labels= c("Female","Male"))) %>%
      within(gender_factor <- relevel(gender_factor, ref = 2)) %>%
      mutate(score_factor = factor(score_text != "Low", labels = c("LowScore","HighScore")))
model <- glm(score_factor ~ gender_factor + age_factor + race_factor +
                            priors_count + crime_factor, family="binomial", data=df)
summary(model)

In [ ]:
%%R
# Convert the log-odds coefficient into "X times as likely" using ProPublica's formula.
# Coefficients come straight from the fitted model, so nothing is retyped by hand.
b <- coef(model)
control <- plogis(b[["(Intercept)"]])   # probability of a higher score for the reference group
rel <- function(term) exp(b[[term]]) / (1 - control + control * exp(b[[term]]))
rel("gender_factorFemale")

In [ ]:
%%R
suppressPackageStartupMessages(library(car))
vif(model <- glm(score_factor ~ gender_factor + age_factor + race_factor +
                            priors_count + crime_factor + two_year_recid, family="binomial", data=df))

## 11. ROC curves

In [ ]:
%%R -w 900 -h 900 -u px
suppressPackageStartupMessages(library(pROC))
raw_data <- read.csv("./compas-scores-two-years.csv")
w <- filter(raw_data, sex == "Female")
wroc <- roc(two_year_recid ~ decile_score, quiet=TRUE, data=w)
m <- filter(raw_data, sex == "Male")
mroc <- roc(two_year_recid ~ decile_score, quiet=TRUE, data=m)
print(wroc); print(mroc)
plot(mroc)
plot(wroc, add=TRUE, col="red")
legend("bottomright", legend = c("Men", "Women"), col = c("black", "red"), lwd = 2)

In [ ]:
%%R -w 900 -h 900 -u px
wroc <- roc(two_year_recid ~ decile_score, quiet=TRUE, data=w, smooth=TRUE)
mroc <- roc(two_year_recid ~ decile_score, quiet=TRUE, data=m, smooth=TRUE)
plot(mroc)
plot(wroc, col="red", add=TRUE)
legend("bottomright", legend = c("Men", "Women"), col = c("black", "red"), lwd = 2)

## 12. Error rate summary

In [ ]:
import pandas as pd

def rates(group):
    r = list(filter(group, recid))
    s = list(filter(group, surv))
    return {"Labeled higher risk, but didn't re-offend": round(count(lambda i: i.high, s) / len(s) * 100, 1),
            "Labeled lower risk, yet did re-offend":     round(count(lambda i: i.low, r) / len(r) * 100, 1)}

pd.DataFrame({"Women": rates(is_female), "Men": rates(is_male)})

## 13. Forest plot

In [ ]:
%%R -w 900 -h 420 -u px
data <- filter(filter(read.csv("./cox-parsed.csv"), score_text != "N/A"), end > start) %>%
        mutate(score_factor = factor(score_text)) %>%
        within(score_factor <- relevel(score_factor, ref=2)) %>%
        mutate(gender_factor = factor(sex, labels= c("Female","Male"))) %>%
        within(gender_factor <- relevel(gender_factor, ref = 2))
gmodel <- coxph(Surv(start, end, event, type="counting") ~ gender_factor * score_factor, data=data)
b <- coef(gmodel); V <- vcov(gmodel)

# hazard ratio and 95% CI for a sum of coefficients
combo <- function(terms) {
  est <- sum(b[terms]); se <- sqrt(sum(V[terms, terms]))
  c(hr = exp(est), lo = exp(est - 1.96 * se), hi = exp(est + 1.96 * se))
}
F <- "gender_factorFemale"; H <- "score_factorHigh"; M <- "score_factorMedium"
FH <- "gender_factorFemale:score_factorHigh"; FM <- "gender_factorFemale:score_factorMedium"

rows <- rbind(
  data.frame(panel = "Women vs. men, same label", row = "Low",              t(combo(F))),
  data.frame(panel = "Women vs. men, same label", row = "Medium",           t(combo(c(F, FM)))),
  data.frame(panel = "Women vs. men, same label", row = "High",             t(combo(c(F, FH)))),
  data.frame(panel = "Label vs. Low, within sex", row = "Men: Medium",      t(combo(M))),
  data.frame(panel = "Label vs. Low, within sex", row = "Women: Medium",    t(combo(c(M, FM)))),
  data.frame(panel = "Label vs. Low, within sex", row = "Men: High",        t(combo(H))),
  data.frame(panel = "Label vs. Low, within sex", row = "Women: High",      t(combo(c(H, FH))))
)
rows$row <- factor(rows$row, levels = rev(unique(rows$row)))
rows$panel <- factor(rows$panel, levels = unique(rows$panel))
rows$sex <- ifelse(grepl("Men", rows$row), "Men", "Women")
print(rows, digits = 3)

ggplot(rows, aes(x = hr, y = row, xmin = lo, xmax = hi, colour = sex)) +
  geom_vline(xintercept = 1, linetype = "dashed", colour = "grey50") +
  geom_pointrange() +
  scale_x_log10() +
  scale_colour_manual(values = c(Men = "grey30", Women = "firebrick")) +
  facet_wrap(~ panel, scales = "free") +
  labs(x = "Hazard ratio (log scale)", y = NULL, colour = NULL,
       title = "COMPAS labels and re-arrest, by sex") +
  theme_minimal(base_size = 14)